# 实践 1.1  从样本表到可建模数据

《统计遗传与智能育种学》第一章 · 电子资源 `01_data_reproducibility.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：01_data_reproducibility.ipynb。实践使用固定随机种子生成一个小型基因型—表型数据集，完成样本主键检查、宽表与长表转换、缺失值统计、基因型矩阵对齐和软件版本记录。本练习重点检查数据结构，为后续模型分析准备准确的输入数据。

运行检查：重启内核并运行全部单元（Restart Kernel and Run All）  
预期：所有样本主键唯一；基因型与表型样本顺序一致；输出缺失率统计与软件版本信息。

## 输入、完整代码、实际输出与解释

输入与可复现条件：8个模拟样本、6个标记及1个显式缺失基因型；SEED=101；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=101
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 101


In [2]:
samples=pd.DataFrame({'sample_id':[f'S{i:02d}' for i in range(1,9)],'population':['A']*4+['B']*4,'phenotype':rng.normal(10,2,8)})
markers=[f'M{i:02d}' for i in range(1,7)]
geno=pd.DataFrame(rng.binomial(2,.35,(8,6)),index=samples.sample_id,columns=markers).astype(float)
geno.loc['S03','M04']=np.nan
print('sample key unique:',samples.sample_id.is_unique)
print('genotype missing cells:',int(geno.isna().sum().sum()))
print('missing rate by marker:')
print(geno.isna().mean().rename('missing_rate').to_string())
print(samples.to_string(index=False))

# 故意打乱基因型顺序，再用主键显式对齐
g_shuffled=geno.sample(frac=1,random_state=SEED)
assert set(samples.sample_id)==set(g_shuffled.index)
g_aligned=g_shuffled.loc[samples.sample_id]
long=g_aligned.reset_index(names='sample_id').melt(id_vars='sample_id',var_name='marker',value_name='dosage')
wide=long.pivot(index='sample_id',columns='marker',values='dosage').loc[samples.sample_id]
print('aligned order:',list(g_aligned.index))
print('long rows:',len(long),'round-trip equal:',wide.equals(g_aligned))
print(long.head(10).to_string(index=False))


sample key unique: True
genotype missing cells: 1
missing rate by marker:
M01    0.000
M02    0.000
M03    0.000
M04    0.125
M05    0.000
M06    0.000
sample_id population  phenotype
      S01          A   8.419695
      S02          A   5.930749
      S03          A  11.206603
      S04          A  11.488589
      S05          B   9.380626
      S06          B  10.734643
      S07          B  13.420789
      S08          B  12.121596
aligned order: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08']
long rows: 48 round-trip equal: True
sample_id marker  dosage
      S01    M01     2.0
      S02    M01     2.0
      S03    M01     1.0
      S04    M01     1.0
      S05    M01     2.0
      S06    M01     1.0
      S07    M01     0.0
      S08    M01     1.0
      S01    M02     0.0
      S02    M02     1.0


结果解释：样本主键唯一且基因型—表型按sample_id显式对齐；M04缺失率为0.125，长表48行可无损回到宽表。本例检查数据结构与样本对应关系，未评价统计模型性能。

可修改练习：把表型表中的S03改成SX3，先做集合差异再尝试对齐；说明为什么不能依赖两个文件当前行顺序。